### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="lung_cancer",
    dataset_year="2001",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Other", # supplementary materials of the paper
    original_dataset_source_download_link="https://www.pnas.org/doi/10.1073/pnas.191502998#supplementary-materials",
    download_description=r"""
Cannot download automatically!

Click and download yourself: "https://www.pnas.org/doi/suppl/10.1073/pnas.191502998/suppl_file/dataseta_12600gene.xls"
mkdir -p local-data-warehouse/lung_cancer/ && mv dataseta_12600gene.xls local-data-warehouse/lung_cancer/
""",
    # References
    academic_reference_bibtex=r"""@article{bhattacharjee2001classification,
  title={Classification of human lung carcinomas by mRNA expression profiling reveals distinct adenocarcinoma subclasses},
  author={Bhattacharjee, Arindam and Richards, William G and Staunton, Jane and Li, Cheng and Monti, Stefano and Vasa, Priya and Ladd, Christine and Beheshti, Javad and Bueno, Raphael and Gillette, Michael and others},
  journal={Proceedings of the National Academy of Sciences},
  volume={98},
  number={24},
  pages={13790--13795},
  year={2001},
  publisher={The National Academy of Sciences}
}
""",
    academic_reference_bibtex_key="bhattacharjee2001classification",
    license="None", # not given as part of data download on the website, likely Copyright for the journal
    data_tags=["IID"],
    curation_comments="""
- The original data comes with lung adenocarcinoma and other adenocarcinomas merged into one class already. We keep the same and were not able to find a way to reverse this from the public data.
- We also drop the class "small-cell lung carcinoma" as it only has 6 samples, which is not a meaningful sample size for benchmarking, or the predictive task.  We could add another version with all classes in the future.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="CancerType",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="CancerType",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_excel(f"{dataset_mold.path}/dataseta_12600gene.xls")

# Translate into tabular format
df = (
    df
    .set_index("probe set")        # use probe set as row index → becomes columns after .T
    .drop(columns=["gene"])        # drop unwanted column
    .T                             # transpose
    .reset_index(names="CancerType")
)
# Extract classes from index
prefix_to_class = {
    "AD": "lung adenocarcinoma",
    "SQ": "squamous cell carcinoma",
    "COID": "pulmonary carcinoid",
    "SMCL": "small-cell lung carcinoma",
    "NL": "normal lung"
}
# extract prefix and map to class
df["CancerType"] = df["CancerType"].str.split("-").str[0].map(prefix_to_class)

df = df[df["CancerType"] != "small-cell lung carcinoma"]
df["CancerType"] = df["CancerType"].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary.head(100) # limit number of cols we show...

In [ ]:
# Numeric Feature Statistics
numeric_stats.head(100) # limit number of cols we show...

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()